# Fase 2 – La ricerca regge nel browser?

Una versione statica dell'app deve fare tutto nel browser: niente bge-m3 (troppo grande) e niente reranker.
Questo notebook misura, sul set di sviluppo e nella lingua scelta, quanto si perde con:

- **solo BM25** (con boost sui codici e allegato implicito);
- **BM25 + un modello di embedding piccolo** che esiste anche in versione browser (transformers.js).

I riferimenti con **bge-m3** (con e senza reranker) vengono ricalcolati qui, sulla stessa lingua.

**Impostazioni:** GPU T4 (serve solo a calcolare in fretta gli embeddings dei chunk), Internet ON, Dataset del parsing.

In [ ]:
# Pacchetto dal repository GitHub.
# Se il repository non e' ancora aggiornato: carica la cartella del progetto come Dataset Kaggle
# e usa invece:  !pip install -q "/kaggle/input/<nome-dataset>/rag-qrt[eval]"
# Dopo un aggiornamento del codice su GitHub: aggiungi --force-reinstall --no-deps e riavvia il kernel.
!pip install -q "qrt-rag[eval] @ git+https://github.com/mat126/rag-qrt-SolvencyII.git"

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

from qrt_rag.chunking import build_chunks
from qrt_rag.evaluation import eval_retrieval, load_eval_set, summarize
from qrt_rag.index import embed_chunks, load_embedder
from qrt_rag.parsing import load_pages
from qrt_rag.retrieval import Retriever, load_reranker

LANG = "en"   # "en" oppure "it"
WORK = Path("/kaggle/working")
PARSED_INPUT = Path("/kaggle/input/datasets/mat126/parsing-rag-qrt-en")   # parsing della stessa lingua
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Modelli candidati: tutti hanno una versione ONNX per transformers.js (organizzazione Xenova).
BGE_EN_Q = "Represent this sentence for searching relevant passages: "
CANDIDATES = {
    "en": {
        "bge-small-en-v1.5": dict(hf="BAAI/bge-small-en-v1.5", q_prefix=BGE_EN_Q, p_prefix=""),
        "all-MiniLM-L6-v2": dict(hf="sentence-transformers/all-MiniLM-L6-v2", q_prefix="", p_prefix=""),
        "multilingual-e5-small": dict(hf="intfloat/multilingual-e5-small",
                                      q_prefix="query: ", p_prefix="passage: "),
        # piu' pesante (circa 3 volte bge-small): quanto vale un modello piu' grande
        "bge-base-en-v1.5": dict(hf="BAAI/bge-base-en-v1.5", q_prefix=BGE_EN_Q, p_prefix=""),
    },
    "it": {
        "multilingual-e5-small": dict(hf="intfloat/multilingual-e5-small",
                                      q_prefix="query: ", p_prefix="passage: "),
        "paraphrase-multilingual-MiniLM-L12-v2": dict(
            hf="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", q_prefix="", p_prefix=""),
        "multilingual-e5-base": dict(hf="intfloat/multilingual-e5-base",
                                     q_prefix="query: ", p_prefix="passage: "),
    },
}[LANG]
print("Lingua:", LANG, "| dispositivo:", DEVICE)

In [ ]:
pages = load_pages(PARSED_INPUT)
chunks = build_chunks(pages, lang=LANG)
cases = load_eval_set(lang=LANG)
print(f"Chunk: {len(chunks)} | casi nel set: {len(cases)}")

In [ ]:
class PrefixedEncoder:
    """Aggiunge il prefisso alle domande (es. "query: " per e5), con la stessa
    interfaccia encode() di SentenceTransformer usata dal Retriever."""
    def __init__(self, model, prefix=""):
        self.model, self.prefix = model, prefix

    def encode(self, texts, **kwargs):
        return self.model.encode([self.prefix + t for t in texts], **kwargs)


def embed_passages(model, prefix, batch_size=64):
    t0 = time.time()
    E = model.encode([prefix + c["text"] for c in chunks], batch_size=batch_size,
                     normalize_embeddings=True, convert_to_numpy=True,
                     show_progress_bar=True).astype("float32")
    print(f"  embeddings {E.shape} in {time.time() - t0:.0f}s")
    return E


results, per_case = [], {}

def record(label, retriever, E=None):
    df = eval_retrieval(retriever, cases)
    row = summarize(df, label)
    if E is not None:
        row["dim"], row["MB float32"] = E.shape[1], round(E.nbytes / 1e6, 1)
    results.append(row)
    per_case[label] = df

# Riferimenti con bge-m3 (l'app attuale)
emb = load_embedder()
E_ref = embed_chunks(emb, chunks)
record("rif.: BM25 + bge-m3", Retriever(chunks, E_ref, emb, lang=LANG), E_ref)
record("rif.: BM25 + bge-m3 + reranker",
       Retriever(chunks, E_ref, emb, reranker=load_reranker(), lang=LANG), E_ref)
del emb
torch.cuda.empty_cache()

# Solo BM25
record("solo BM25", Retriever(chunks, lang=LANG))

# BM25 + ciascun modello piccolo (senza reranker, come nel browser)
for name, cfg in CANDIDATES.items():
    print("==", name)
    model = SentenceTransformer(cfg["hf"], device=DEVICE)
    print("  lunghezza massima:", model.max_seq_length, "token")
    E = embed_passages(model, cfg["p_prefix"])
    np.save(WORK / f"embeddings_{LANG}_{name}.npy", E)
    record(f"BM25 + {name}", Retriever(chunks, E, PrefixedEncoder(model, cfg["q_prefix"]), lang=LANG), E)
    del model
    torch.cuda.empty_cache()

summary = pd.DataFrame(results).round(3)
display(summary)
summary.to_csv(WORK / f"fase2_confronto_{LANG}.csv", index=False)

In [ ]:
# Dove si perde: posizione della fonte giusta per ogni domanda e configurazione
# (NaN = oltre il 20esimo posto)
table = pd.DataFrame({name: df.set_index("id")["rank"] for name, df in per_case.items()})
table.insert(0, "tipo", per_case["solo BM25"].set_index("id")["tipo"])
display(table)

print("hit@5 per tipo di domanda:")
display(table.groupby("tipo").agg(lambda col: (col.astype(float) <= 5).mean()).round(2))
table.to_csv(WORK / f"fase2_per_domanda_{LANG}.csv")

In [ ]:
# Tempo per codificare una domanda su CPU: indicativo di quanto servira' nel browser
# (transformers.js in WebAssembly e' di solito piu' lento di PyTorch su CPU)
domande = [c["q"] for c in cases[:20]]
for name, cfg in CANDIDATES.items():
    m = SentenceTransformer(cfg["hf"], device="cpu")
    m.encode([cfg["q_prefix"] + domande[0]])              # riscaldamento
    t0 = time.time()
    for q in domande:
        m.encode([cfg["q_prefix"] + q], normalize_embeddings=True)
    print(f"{name:<40} {1000 * (time.time() - t0) / len(domande):.0f} ms per domanda su CPU")
    del m

# Dimensione dei chunk da scaricare nel browser (solo i campi necessari)
light = [{k: c[k] for k in ("id", "kind", "page", "annex", "article", "template",
                            "template_full", "cells", "text")} for c in chunks]
raw = json.dumps(light, ensure_ascii=False).encode("utf-8")
print(f"\nChunk in JSON: {len(raw) / 1e6:.1f} MB (compressi gzip circa un quarto)")